# Notebook 03 — Retrieval Debug
Deep-dive into retrieval behaviour: compare V1/V2/V3 on the same queries,
inspect BM25 vs dense results, and identify misses and false positives.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd

DEBUG_QUERIES = [
    'What was Grab total revenue for FY2023?',
    'What are the key risks from regulation in Southeast Asia?',
    'How did adjusted EBITDA change from FY2022 to FY2023?',
    'What long term GMV targets did Grab announce at Investor Day 2023?',
    'What was Grab cash position at end of FY2023?',
]

def retrieve_all_variants(query):
    results = {}
    try:
        from src.retrieval.dense_retriever import DenseRetriever
        results['dense'] = DenseRetriever().retrieve(query, top_k=5)
    except Exception as e:
        results['dense'] = []
        print(f'Dense failed: {e}')

    try:
        from src.retrieval.sparse_retriever import SparseRetriever
        results['sparse'] = SparseRetriever().retrieve(query, top_k=5)
    except Exception as e:
        results['sparse'] = []
        print(f'Sparse failed: {e}')

    try:
        from src.retrieval.hybrid_retriever import HybridRetriever
        results['hybrid'] = HybridRetriever().retrieve(query)[:5]
    except Exception as e:
        results['hybrid'] = []
        print(f'Hybrid failed: {e}')

    return results

In [ ]:
# Side-by-side comparison for one query
query = DEBUG_QUERIES[0]
print(f'Query: {query}')
print('='*80)

results = retrieve_all_variants(query)

for method, chunks in results.items():
    print(f'\n--- {method.upper()} Top 5 ---')
    for i, c in enumerate(chunks[:5], 1):
        m = c.get('metadata', {})
        score = c.get('rerank_score', c.get('rrf_score', c.get('score', 0)))
        print(f'  {i}. [{score:.4f}] {m.get("doc_type","")} | {m.get("fiscal_period","")} | p{m.get("page_number","")} | {m.get("chunk_id","")}')
        print(f'     {c["text"][:120]}...')

In [ ]:
# Overlap analysis: which chunks appear in both dense and sparse top-5?
dense_ids = {c['metadata'].get('chunk_id') for c in results.get('dense', [])}
sparse_ids = {c['metadata'].get('chunk_id') for c in results.get('sparse', [])}

overlap = dense_ids & sparse_ids
only_dense = dense_ids - sparse_ids
only_sparse = sparse_ids - dense_ids

print(f'Overlap (in both): {len(overlap)} → {overlap}')
print(f'Dense only: {len(only_dense)} → {only_dense}')
print(f'Sparse only: {len(only_sparse)} → {only_sparse}')

In [ ]:
# Load and inspect retrieval logs
import json
from src.utils.config_loader import load_config
cfg = load_config()
log_dir = Path(cfg['paths'].get('retrieval_logs', 'indexes/retrieval_logs'))

for log_file in log_dir.glob('*.jsonl'):
    print(f'\n=== {log_file.name} ===')
    with open(log_file) as f:
        lines = f.readlines()[-5:]  # Last 5 entries
    for line in lines:
        entry = json.loads(line)
        print(f"  {entry['timestamp'][:19]} | latency={entry['latency_ms']:.0f}ms | n={entry['num_returned']} | Q: {entry['query'][:50]}")